# N8N Workflow Downloader — Widgets

Versión con `ipywidgets` del downloader de workflows live.

Si tu entorno de notebook renderiza widgets bien, esta es la versión más cómoda.
Si ves `Error displaying widget: model not found`, usa la otra:
[N8N Workflow Downloader.ipynb](/Users/fernandorau/Documents/New%20project/notebooks/N8N%20Workflow%20Downloader.ipynb)


In [1]:
from __future__ import annotations

import json
import re
import urllib.error
import urllib.parse
import urllib.request
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import ipywidgets as widgets
from IPython.display import Markdown, display, clear_output

DEFAULT_BASE_URL = 'https://n8n.wasilda.cloud'
DEFAULT_OUTDIR = str(Path('/Users/fernandorau/Documents/New project/workflows-live'))

PRESETS = {
    'none': [],
    'polanco-audit': [
        {'id': '2iW6U7DC1f5T8oeT', 'label': 'gateway'},
        {'id': 'zdoIzfwKegvB9YN4', 'label': 'appointment_handler'},
        {'id': 'p1cC2oPoQ9C4kUWX', 'label': 'booking_handler'},
        {'id': '6Pow6lkluNNYOgCK', 'label': 'session_cleanup'},
        {'id': 'l2C3hjJWBKtGOe8A', 'label': 'info_handler'},
        {'id': 'OfycAHdi4b6Ske7z', 'label': 'chatwoot_inbound_mirror'},
        {'id': '7mbhs9tVxu9vxrSE', 'label': 'chatwoot_reply_bridge'},
        {'id': 'snqVw0DuCqJEf2gx', 'label': 'dentegra_pipeline'},
        {'id': 'Yf1aZiMuBxnSxWcL', 'label': 'cmp_patient_creator'},
        {'id': 'KsheY85rfGQoTbSC', 'label': 'cmp_appointment_booker'},
        {'id': 'QWca5ece2OxXALvI', 'label': 'cmp_confirm_direct'},
        {'id': 'BJvKQnKRxP4sm4VK', 'label': 'cmp_cancel_direct'},
    ],
    'polanco-booking': [
        {'id': '2iW6U7DC1f5T8oeT', 'label': 'gateway'},
        {'id': 'p1cC2oPoQ9C4kUWX', 'label': 'booking_handler'},
        {'id': 'zdoIzfwKegvB9YN4', 'label': 'appointment_handler'},
        {'id': 'Yf1aZiMuBxnSxWcL', 'label': 'cmp_patient_creator'},
        {'id': 'KsheY85rfGQoTbSC', 'label': 'cmp_appointment_booker'},
        {'id': 'QWca5ece2OxXALvI', 'label': 'cmp_confirm_direct'},
        {'id': 'BJvKQnKRxP4sm4VK', 'label': 'cmp_cancel_direct'},
        {'id': 'snqVw0DuCqJEf2gx', 'label': 'dentegra_pipeline'},
    ],
    'polanco-chatwoot': [
        {'id': '2iW6U7DC1f5T8oeT', 'label': 'gateway'},
        {'id': 'OfycAHdi4b6Ske7z', 'label': 'chatwoot_inbound_mirror'},
        {'id': '7mbhs9tVxu9vxrSE', 'label': 'chatwoot_reply_bridge'},
        {'id': 'l2C3hjJWBKtGOe8A', 'label': 'info_handler'},
    ],
}


@dataclass
class ApiClient:
    base_url: str
    api_key: str
    timeout: int = 30

    def request(self, path: str, params: dict | None = None):
        query = ''
        if params:
            cleaned = {k: v for k, v in params.items() if v not in (None, '')}
            if cleaned:
                query = '?' + urllib.parse.urlencode(cleaned, doseq=True)
        url = self.base_url.rstrip('/') + path + query
        req = urllib.request.Request(
            url,
            headers={
                'X-N8N-API-KEY': self.api_key,
                'Accept': 'application/json',
                'User-Agent': 'codex-workflow-downloader-widgets/1.0',
            },
        )
        try:
            with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                payload = resp.read().decode('utf-8')
                return json.loads(payload)
        except urllib.error.HTTPError as exc:
            body = exc.read().decode('utf-8', errors='replace')
            raise RuntimeError(f'HTTP {exc.code} for {url}\n{body[:1200]}') from exc
        except urllib.error.URLError as exc:
            raise RuntimeError(f'Network error for {url}: {exc}') from exc

    def get_workflow(self, workflow_id: str):
        return self.request(f'/api/v1/workflows/{workflow_id}')

    def list_tags(self):
        """Fetch all tags from /api/v1/tags."""
        payload = self.request('/api/v1/tags')
        return extract_items(payload)

    def resolve_tag_to_name(self, tag_filter: str):
        """Resolve a tag filter (name or ID) to the actual tag name."""
        needle = normalize_tag_token(tag_filter)
        if not needle:
            return None
        try:
            all_tags = self.list_tags()
        except Exception:
            return tag_filter.strip()
        for tag in all_tags:
            if normalize_tag_token(tag.get('name', '')) == needle:
                return tag['name']
            if normalize_tag_token(str(tag.get('id', ''))) == needle:
                return tag['name']
        return tag_filter.strip()

    def list_workflows_page(self, limit: int = 250, cursor: str = '', tags: str = ''):
        params = {'limit': limit}
        if cursor:
            params['cursor'] = cursor
        if tags:
            params['tags'] = tags
        return self.request('/api/v1/workflows', params)

    def list_all_workflows(self, limit: int = 250, tags: str = ''):
        results = []
        cursor = ''
        while True:
            payload = self.list_workflows_page(limit=limit, cursor=cursor, tags=tags)
            page_items = extract_items(payload)
            results.extend(page_items)
            cursor = payload.get('nextCursor') or payload.get('cursor') or ''
            if not cursor or not page_items:
                break
        return results


def extract_items(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ('data', 'items', 'results'):
            value = payload.get(key)
            if isinstance(value, list):
                return value
    return []


def sanitize(value: str, max_len: int = 100) -> str:
    value = value or 'unnamed'
    value = re.sub(r'[^\w\s.-]+', '-', value, flags=re.UNICODE)
    value = re.sub(r'\s+', ' ', value).strip().replace(' ', '_')
    value = re.sub(r'_+', '_', value)
    return value[:max_len] or 'unnamed'


def unique_preserve(items):
    seen = set()
    out = []
    for item in items:
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out


def parse_workflow_ids(text: str):
    raw = re.split(r'[\s,]+', text or '')
    return unique_preserve([part.strip() for part in raw if part.strip()])


def normalize_tag_token(value: str) -> str:
    return str(value or '').strip().lower()


def canonical_workflow_name(value: str) -> str:
    return re.sub(r'\s+', ' ', str(value or '').strip()).lower()


def parse_dt(value: str):
    raw = str(value or '').strip()
    if not raw:
        return datetime.min
    try:
        return datetime.fromisoformat(raw.replace('Z', '+00:00'))
    except Exception:
        return datetime.min


def workflow_tags(workflow: dict):
    tags = workflow.get('tags') or []
    out = []
    for tag in tags:
        if isinstance(tag, dict):
            out.append({'id': str(tag.get('id', '')).strip(), 'name': str(tag.get('name', '')).strip()})
        elif isinstance(tag, str):
            out.append({'id': '', 'name': tag.strip()})
    return out


def workflow_matches_tag(workflow: dict, tag_filter: str) -> bool:
    needle = normalize_tag_token(tag_filter)
    if not needle:
        return False
    for tag in workflow_tags(workflow):
        if normalize_tag_token(tag['name']) == needle:
            return True
        if normalize_tag_token(tag['id']) == needle:
            return True
    return False


def apply_metadata_filters(workflows, active_only_flag=False, latest_only_flag=False, name_contains_value=''):
    filtered = list(workflows)
    if active_only_flag:
        filtered = [wf for wf in filtered if wf.get('active') is True]
    needle = str(name_contains_value or '').strip().lower()
    if needle:
        filtered = [wf for wf in filtered if needle in str(wf.get('name', '')).lower()]
    if latest_only_flag:
        latest = {}
        for wf in filtered:
            key = canonical_workflow_name(wf.get('name', '') or wf.get('id', ''))
            prev = latest.get(key)
            if prev is None or parse_dt(wf.get('updatedAt', '')) > parse_dt(prev.get('updatedAt', '')):
                latest[key] = wf
        filtered = list(latest.values())
    filtered.sort(key=lambda wf: (canonical_workflow_name(wf.get('name', '') or wf.get('id', '')), str(wf.get('id', ''))))
    return filtered


base_url = widgets.Text(value=DEFAULT_BASE_URL, description='Base URL:', layout=widgets.Layout(width='900px'), style={'description_width': '120px'})
api_key = widgets.Password(value='', description='API key:', placeholder='Pega aquí el API key live', layout=widgets.Layout(width='900px'), style={'description_width': '120px'})
preset = widgets.Dropdown(options=list(PRESETS.keys()), value='polanco-audit', description='Preset:', layout=widgets.Layout(width='420px'), style={'description_width': '120px'})
workflow_ids = widgets.Textarea(value='', description='Workflow IDs:', placeholder='Uno por línea o separados por comas', layout=widgets.Layout(width='900px', height='120px'), style={'description_width': '120px'})
tag_filter = widgets.Text(value='', description='Tag filter:', placeholder='Nombre o id del tag', layout=widgets.Layout(width='900px'), style={'description_width': '120px'})
active_only = widgets.Checkbox(value=False, description='Active only', indent=False)
latest_per_name_only = widgets.Checkbox(value=False, description='Latest per name only', indent=False)
name_contains = widgets.Text(value='', description='Name contains:', placeholder='Substring opcional', layout=widgets.Layout(width='900px'), style={'description_width': '120px'})
outdir = widgets.Text(value=DEFAULT_OUTDIR, description='Out dir:', layout=widgets.Layout(width='900px'), style={'description_width': '120px'})
bundle_name = widgets.Text(value='', description='Bundle name:', placeholder='Opcional. Si lo dejas vacío se usa timestamp', layout=widgets.Layout(width='900px'), style={'description_width': '120px'})
list_button = widgets.Button(description='List Matches', button_style='warning', icon='tags')
download_button = widgets.Button(description='Download Workflows', button_style='success', icon='download')
output = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='12px'))


def build_client():
    return ApiClient(base_url.value.strip(), api_key.value.strip())


def collect_requested_and_filtered():
    client = build_client()
    requested = []
    for item in PRESETS.get(preset.value, []):
        requested.append({'id': item['id'], 'label': item.get('label', ''), 'source': 'preset'})
    for wid in parse_workflow_ids(workflow_ids.value):
        requested.append({'id': wid, 'label': '', 'source': 'manual'})

    tag_matches = []
    if tag_filter.value.strip():
        tag_value = tag_filter.value.strip()
        # Try server-side filtering first to reduce the candidate set
        resolved_name = client.resolve_tag_to_name(tag_value)
        candidates = []
        if resolved_name:
            candidates = client.list_all_workflows(limit=250, tags=resolved_name)
        # If server-side returned nothing, get all workflows as candidates
        if not candidates:
            candidates = client.list_all_workflows(limit=250)
        # ALWAYS verify tags by fetching each workflow individually.
        # The list endpoint does not include tags in the response, and
        # the server-side tags parameter is unreliable in many n8n versions.
        for wf_summary in candidates:
            wid = str(wf_summary.get('id', '')).strip()
            if not wid:
                continue
            full_wf = client.get_workflow(wid)
            if workflow_matches_tag(full_wf, tag_value):
                tag_matches.append(full_wf)
        for wf in tag_matches:
            requested.append({'id': str(wf.get('id', '')).strip(), 'label': '', 'source': 'tag'})

    dedup = []
    seen = set()
    for item in requested:
        wid = item['id']
        if wid and wid not in seen:
            seen.add(wid)
            dedup.append(item)

    cache = {}
    metadata = []
    for item in dedup:
        wf = client.get_workflow(item['id'])
        cache[item['id']] = wf
        metadata.append({
            'id': item['id'],
            'label': item.get('label', ''),
            'source': item.get('source', ''),
            'name': wf.get('name', ''),
            'active': wf.get('active', False),
            'tags': workflow_tags(wf),
            'createdAt': wf.get('createdAt', ''),
            'updatedAt': wf.get('updatedAt', ''),
        })

    selected = apply_metadata_filters(
        metadata,
        active_only_flag=active_only.value,
        latest_only_flag=latest_per_name_only.value,
        name_contains_value=name_contains.value,
    )
    return client, cache, tag_matches, selected


def list_matches(_):
    with output:
        clear_output()
        try:
            client, cache, tag_matches, selected = collect_requested_and_filtered()
        except Exception as exc:
            print(exc)
            return
        print('Preset:', preset.value)
        print('Tag filter:', tag_filter.value.strip() or '(none)')
        print('Active only:', active_only.value)
        print('Latest per name only:', latest_per_name_only.value)
        print('Name contains:', name_contains.value.strip() or '(none)')
        print('Raw tag matches:', len(tag_matches))
        print('Selected after filters:', len(selected))
        print()
        for wf in selected[:120]:
            tag_names = ', '.join([t['name'] for t in wf['tags'] if t['name']])
            print(f"- {wf['name']} (ID: {wf['id']}) | active={wf['active']} | updatedAt={wf['updatedAt']} | source={wf['source']} | tags=[{tag_names}]")


def download_workflows(_):
    with output:
        clear_output()
        try:
            client, cache, tag_matches, selected = collect_requested_and_filtered()
        except Exception as exc:
            print(exc)
            return

        if not selected:
            print('No hay workflows seleccionados después de los filtros.')
            return

        bundle_root = Path(outdir.value).expanduser()
        final_bundle = bundle_name.value.strip() or datetime.now().strftime('%Y-%m-%d__%H-%M-%S')
        bundle_dir = bundle_root / final_bundle
        workflows_dir = bundle_dir / 'workflows'
        workflows_dir.mkdir(parents=True, exist_ok=True)

        summary_rows = ['id\tname\tactive\ttags\tcreatedAt\tupdatedAt\tsource\tfile']
        manifest = {
            'generatedAt': datetime.now().isoformat(),
            'baseUrl': base_url.value.strip(),
            'preset': preset.value,
            'tagFilter': tag_filter.value.strip() or None,
            'filters': {
                'activeOnly': active_only.value,
                'latestPerNameOnly': latest_per_name_only.value,
                'nameContains': name_contains.value.strip() or None,
            },
            'requestedWorkflowIds': [wf['id'] for wf in selected],
            'workflows': [],
        }

        for index, item in enumerate(selected, start=1):
            wf = cache[item['id']]
            file_name = f"workflow-{item['id']}__{sanitize(item['name'] or item['id'])}.json"
            file_path = workflows_dir / file_name
            file_path.write_text(json.dumps(wf, ensure_ascii=False, indent=2), encoding='utf-8')
            tag_names = ', '.join([t['name'] for t in item['tags'] if t['name']])
            summary_rows.append(f"{item['id']}\t{item['name']}\t{item['active']}\t{tag_names}\t{item['createdAt']}\t{item['updatedAt']}\t{item['source']}\t{file_path}")
            manifest['workflows'].append({
                'id': item['id'],
                'label': item.get('label', ''),
                'source': item.get('source', ''),
                'name': item['name'],
                'active': item['active'],
                'tags': item['tags'],
                'createdAt': item['createdAt'],
                'updatedAt': item['updatedAt'],
                'file': str(file_path),
            })
            print(f"[{index}/{len(selected)}] saved {item['id']} -> {item['name']}")

        ids_path = bundle_dir / 'ids.txt'
        ids_path.write_text('\n'.join([wf['id'] for wf in selected]) + '\n', encoding='utf-8')
        summary_path = bundle_dir / 'workflow-summary.tsv'
        summary_path.write_text('\n'.join(summary_rows) + '\n', encoding='utf-8')
        manifest_path = bundle_dir / 'manifest.json'
        manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
        print('\nDone.')
        print('ids.txt:', ids_path)
        print('workflow-summary.tsv:', summary_path)
        print('manifest.json:', manifest_path)


list_button.on_click(list_matches)
download_button.on_click(download_workflows)

display(Markdown('## Workflow Export Form'))
display(base_url)
display(api_key)
display(preset)
display(workflow_ids)
display(tag_filter)
display(widgets.HBox([active_only, latest_per_name_only]))
display(name_contains)
display(outdir)
display(bundle_name)
display(widgets.HBox([list_button, download_button]))
display(output)


## Workflow Export Form

Text(value='https://n8n.wasilda.cloud', description='Base URL:', layout=Layout(width='900px'), style=TextStyle…

Password(description='API key:', layout=Layout(width='900px'), placeholder='Pega aquí el API key live', style=…

Dropdown(description='Preset:', index=1, layout=Layout(width='420px'), options=('none', 'polanco-audit', 'pola…

Textarea(value='', description='Workflow IDs:', layout=Layout(height='120px', width='900px'), placeholder='Uno…

Text(value='', description='Tag filter:', layout=Layout(width='900px'), placeholder='Nombre o id del tag', sty…

Text(value='', description='Name contains:', layout=Layout(width='900px'), placeholder='Substring opcional', s…

Text(value='/Users/fernandorau/Documents/New project/workflows-live', description='Out dir:', layout=Layout(wi…

Text(value='', description='Bundle name:', layout=Layout(width='900px'), placeholder='Opcional. Si lo dejas va…

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

NewKidOnTheBlock

Preset:NONE

Active only

Latest per name only

API_KEY= 
 





## Tip

Si un tag trae demasiados workflows con el mismo nombre:

- marca `Active only`
- marca `Latest per name only`
- si hace falta, usa `Name contains`
- primero dale `List Matches`
- luego `Download Workflows`
